*0.2 Math / ML basics*

# attention math

**The situation.** A customer writes: *"I ordered a lamp and a rug. It arrived broken."* Which one arrived broken? A person reads "it" and looks back. A transformer does the same, with numbers: for every word it computes how much to *attend* to every other word, and pulls in their information accordingly. That mechanism is attention, and it is three matrix multiplications and one softmax.

**The three matrices.** Each word's vector is turned into a *query* (what am I looking for?), a *key* (what do I contain?) and a *value* (what do I pass on?). Score every query against every key with a dot product, softmax the scores into weights, and average the values with those weights. That is the whole formula.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Run it two ways.** First the formula written out, then PyTorch's production function `scaled_dot_product_attention`, which is what every serving stack calls. Tiny made-up vectors: 5 words, 8 numbers each.

In [2]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)
words = ["lamp", "rug", "it", "arrived", "broken"]
dimension = 8
queries = torch.randn(5, dimension)
keys = torch.randn(5, dimension)
values = torch.randn(5, dimension)

# The formula: softmax(Q · Kᵀ / √d) · V
scores = queries @ keys.T / math.sqrt(dimension)  # every word against every word
weights = torch.softmax(scores, dim=-1)  # each row sums to 1
by_formula = weights @ values

by_pytorch = F.scaled_dot_product_attention(queries, keys, values)
print("formula and PyTorch agree:", torch.allclose(by_formula, by_pytorch, atol=1e-5))
print("output shape:", tuple(by_pytorch.shape), "— one new vector per word")
assert torch.allclose(by_formula, by_pytorch, atol=1e-5)

formula and PyTorch agree: True
output shape: (5, 8) — one new vector per word


**Reading the output.** Same result from the four-line formula and from the optimised library call. Each word comes out as a new vector that mixes in information from the words it attended to.

**Look at the weights for "it".** Random vectors here, so the numbers are not meaningful — but the *shape* of what a trained model produces is exactly this row: a distribution over the other words.

In [3]:
row = weights[words.index("it")]
for word, weight in zip(words, row):
    print(f"it → {word:<8} {weight:.2f}")
print("row sums to:", round(float(row.sum()), 4))
assert abs(float(row.sum()) - 1.0) < 1e-5

it → lamp     0.04
it → rug      0.41
it → it       0.06
it → arrived  0.35
it → broken   0.14
row sums to: 1.0


```
          lamp  rug   it   arrived broken
   it  →  [0.45 0.30 0.10  0.05   0.10 ]   ← trained model: "it" attends mostly to "lamp"
                                             output for "it" = 0.45·V(lamp) + 0.30·V(rug) + …
```

**The rule to remember.** Attention = `softmax(Q Kᵀ / √d) V`. Dot products decide *who to look at*; softmax turns that into shares; the values are averaged by those shares.

| Use it when | Don't when | Instead use |
|---|---|---|
| understanding why context length costs what it does, reading model code, debugging serving | building a model from scratch in production | `torch.nn.MultiheadAttention` or a served model |

**Watch out**
- The score matrix is `words × words`. Doubling the context quadruples the work — that is why long contexts are expensive.
- `√d` is not decoration: without it the scores grow with the dimension, softmax saturates and the model stops learning.
- Real models run many attention "heads" in parallel and stack dozens of layers; the math in each is this.